### Project Airline Assistant along with Multimodals and Custom Blocks using Gradio

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3


In [2]:
load_dotenv(override=True)

GROQ_BASE_URL = os.getenv('GROQ_BASE_URL')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if GROQ_API_KEY:
    print(f'GROQ API KEY found and starts with {GROQ_API_KEY[0:3]}')
else:
    print('GROQ API KEY not found')

groq = OpenAI(base_url = GROQ_BASE_URL, api_key = GROQ_API_KEY)

GROQ API KEY found and starts with gsk


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
DB = "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/002_week/day_4/prices.db"

In [5]:
def get_ticket_price(city):
    print(f'Database Tool Called for the city {city}', flush=True)      # flush=True forces the output to appear immediately
    with sqlite3.connect(DB) as conn:                                   # opens the conncetion with the database
        cursor = conn.cursor()                                          # creates a cursor object to run the SQL commands
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),)) # ? is the placeholder
        result = cursor.fetchone()
    return f'Ticket price to {city} is ${result[0]}' if result else 'No price data available for this city'


#### Note:

- If we don't use flush=True, the print output may be buffered and may not appear immediately in the terminal or notebook

#### What Buffering means:

- Python sometimes holds output in memory for a short time before sending it to the console. This is called buffering.

- So without flush=True:

    - the message may appear later
    - it may not show right away when debugging
    - in some environments, especially long running scripts you may think the function never ran

- With flush=True

    - it forces the Python to send the output immediately
    - useful for debugging or when the program is running in a tool call flow

In [6]:
get_ticket_price('london')

Database Tool Called for the city london


'Ticket price to london is $799.0'

In [7]:
my_tool_schema = {
    "name" : "get_ticket_price",
    "description" : "Get the price of a return ticket to the destination city",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city" : {
                "type" : "string",
                "description" : "The city that customer wants to travel to",
            },
        },
        "required" : ['destination_city'],
        "additionalProperties" : False
    }
}

tools = [{'type':'function', 'function':my_tool_schema}]

tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [8]:
def chatbot(message, history):
    history = [{'role':h['role'], 'content':h['content']} for h in history]
    messages = [{'role':'system', 'content':system_message}] + history + [{'role':'user', 'content':message}]
    response = groq.chat.completions.create(
        model = 'openai/gpt-oss-20b',
        messages = messages,
        tools = tools
    )

    while response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(
            model = 'openai/gpt-oss-20b',
            messages = messages,
            tools = tools
        )
    return response.choices[0].message.content


In [9]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses = [{
                'role':'tool',
                'content':price_details,
                'tool_call_id':tool_call.id
            }]
    return responses


In [ ]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


Database Tool Called for the city London
Database Tool Called for the city Paris


#### What Gradio actually do?

1. Gradio constructs a frontend Svelte app based on our Python description of the UI

2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this Svelte app

3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

4. And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.